
# EchoModel — Pipeline Completo de Construção do Dataset e Treinamento

**Objetivo final do projeto:** construir o **EchoModel**, uma arquitetura de IA para
detecção/classificação de vocalizações de aves, comparável ao **Perch v2** (Google)
e ao **BirdNET v3** (Cornell/Chemnitz), usando metodologia semelhante de dados
(Xeno-Canto + soundscapes anotados).

**Diferencial do EchoModel:** além do rótulo de espécie (target), o modelo também
aprende a prever a **localização temporal (segundos)** e a **localização em
frequência (Hz)** da vocalização dentro do espectrograma de entrada (janelas de
**5 s com overlap de 2,5 s**). A hipótese é que, como cada espécie vocaliza em
faixas de frequência relativamente características, ensinar o modelo a "apontar"
para a região tempo-frequência da vocalização funciona como uma forma de
**atenção supervisionada**, ajudando a classificação.

## Etapas deste notebook

1. **Levantamento de datasets** com rotulagem em **tempo (s) + frequência (Hz)**
   (bounding boxes em espectrograma) — necessários para treinar os YOLOs.
2. **Download e padronização** desses datasets (áudio + anotações).
3. **Conversão** áudio + anotações → **espectrogramas + labels no formato YOLO**.
4. **Treinamento de vários modelos YOLO** (YOLOv8/v9/v10/v11, tamanhos n/s/m) e
   **comparação** de métricas (mAP, precisão, recall) para escolher o melhor
   "detector tempo-frequência".
5. **Aplicação do melhor YOLO** nos datasets usados pelo Perch v2 / BirdNET v3
   (essencialmente Xeno-Canto + soundscapes), extraindo para cada gravação:
   - bounding box temporal (t_min, t_max em segundos)
   - bounding box em frequência (f_min, f_max em Hz)
   - target (espécie), vindo do próprio dataset de origem (1 target por arquivo
     nesta primeira etapa).
6. **Construção do dataset do EchoModel**: janelas de espectrograma de 5 s com
   overlap de 2,5 s, com saída = (bbox temporal relativo à janela, bbox em
   frequência, target).
7. **Arquitetura e treinamento do EchoModel** (multi-task: classificação +
   regressão de bounding box tempo-frequência).
8. **Protocolo de avaliação** comparável ao Perch v2 / BirdNET v3.

> ⚠️ Este notebook é um **pipeline de referência, documentado passo a passo**.
> Os downloads são grandes (gigabytes) e o treinamento dos YOLOs e do EchoModel
> exige GPU. Ajuste os parâmetros (`DOWNLOAD_DIR`, listas de datasets,
> `epochs`, `batch_size`, etc.) ao seu ambiente antes de rodar tudo de uma vez.



## 1. Datasets com rotulagem em tempo (s) **e** frequência (Hz)

A maioria dos dados usados por BirdNET e Perch (Xeno-Canto) é **fracamente
rotulada**: cada gravação tem um rótulo de espécie para o arquivo inteiro, sem
indicar *onde* (tempo/frequência) está a vocalização. Para treinar um detector
tempo-frequência (os YOLOs), precisamos de datasets com **bounding boxes**
(início/fim em segundos + frequência mínima/máxima em Hz) por vocalização.

O dataset que você encontrou (Zenodo **7050014** — *"A collection of
fully-annotated soundscape recordings from the Western United States"*) faz
parte de uma **família de datasets do K. Lisa Yang Center for Conservation
Bioacoustics (Cornell Lab of Ornithology)**, todos no **mesmo formato**:

- Um arquivo `annotations.csv` com colunas:
  `filename, start_time (s), end_time (s), low_freq (Hz), high_freq (Hz), label (código eBird)`
- Gravações de 1 hora em `.flac`/`.wav`.

Esses datasets foram, em boa parte, usados como **dados de teste do BirdCLEF**
(2020–2023), o que os torna muito relevantes — eles cobrem regiões geográficas
diferentes (EUA, Amazônia peruana, Havaí, Colômbia/Costa Rica), aumentando a
diversidade de espécies e condições acústicas para treinar o YOLO.

### Datasets recomendados (mesmo formato `annotations.csv`)

| # | Dataset | Região | Duração | Bboxes | Espécies | Zenodo |
|---|---------|--------|---------|--------|----------|--------|
| 1 | Western United States | Sierra Nevada, EUA | 33 h | 20.147 | 56 | [7050014](https://zenodo.org/records/7050014) |
| 2 | Northeastern United States | Sapsucker Woods, NY, EUA | 285 h | 50.760 | 81 | [7079380](https://zenodo.org/records/7079380) |
| 3 | Southwestern Amazon Basin | Madre de Dios, Peru | 21 h | 14.798 | 132 | [7079124](https://zenodo.org/records/7079124) |
| 4 | Island of Hawai'i | Havaí, EUA | — | — | — | [7078499](https://zenodo.org/records/7078499) |
| 5 | Neotropical Coffee Farms | Colômbia / Costa Rica | 34 h | 6.952 | 89 | [7525349](https://zenodo.org/records/7525349) |
| 6 | Southern Sierra Nevada | Califórnia, EUA | — | — | — | [7525805](https://zenodo.org/records/7525805) |

### Dataset adicional (formato diferente, mas também tempo+frequência)

- **Rainforest Connection (RFCx) Species Audio Detection** (Kaggle, 2021):
  `train_tp.csv` com colunas `recording_id, species_id, songtype_id, t_min,
  f_min, t_max, f_max` — também bounding boxes tempo-frequência, em floresta
  tropical (Porto Rico). Útil para aumentar a diversidade de "formatos" de
  vocalização que o YOLO vê. Requer Kaggle API/credenciais.

### Por que NÃO usar (nesta etapa) certos datasets famosos

- **NIPS4Bplus**, **Cornell Birdcall Identification (2020)**, **Xeno-Canto bruto**:
  fornecem rótulo de espécie e, no máximo, **apenas o tempo** (início/fim) —
  **sem** faixa de frequência. Eles servem para a **Etapa 5** (aplicar o YOLO),
  não para treinar o YOLO.

A célula abaixo cria um **registro (dicionário Python)** com todos os datasets
de bbox tempo-frequência que serão usados para treinar os YOLOs.


In [ ]:

# Registro dos datasets com bounding box tempo-frequência (treino dos YOLOs)
# Cada entrada aponta para o registro Zenodo. O link "files-archive" baixa
# todos os arquivos do registro em um único .zip.

BBOX_DATASETS = {
    "western_us": {
        "zenodo_id": "7050014",
        "name": "Fully-annotated soundscapes - Western United States",
        "n_hours": 33,
        "n_boxes": 20147,
        "n_species": 56,
    },
    "northeastern_us": {
        "zenodo_id": "7079380",
        "name": "Fully-annotated soundscapes - Northeastern United States",
        "n_hours": 285,
        "n_boxes": 50760,
        "n_species": 81,
    },
    "amazon_basin": {
        "zenodo_id": "7079124",
        "name": "Fully-annotated soundscapes - Southwestern Amazon Basin",
        "n_hours": 21,
        "n_boxes": 14798,
        "n_species": 132,
    },
    "hawaii": {
        "zenodo_id": "7078499",
        "name": "Fully-annotated soundscapes - Island of Hawai'i",
        "n_hours": None,
        "n_boxes": None,
        "n_species": None,
    },
    "coffee_farms": {
        "zenodo_id": "7525349",
        "name": "Fully-annotated soundscapes - Neotropical Coffee Farms (CO/CR)",
        "n_hours": 34,
        "n_boxes": 6952,
        "n_species": 89,
    },
    "sierra_nevada_south": {
        "zenodo_id": "7525805",
        "name": "Fully-annotated soundscapes - Southern Sierra Nevada",
        "n_hours": None,
        "n_boxes": None,
        "n_species": None,
    },
}

for key, info in BBOX_DATASETS.items():
    print(f"{key:20s} -> Zenodo {info['zenodo_id']:>8s}  | {info['name']}")



## 2. Configuração do ambiente

Bibliotecas necessárias para todo o pipeline:

- `librosa`, `soundfile` — leitura de áudio e geração de espectrogramas
- `numpy`, `pandas` — manipulação de dados
- `pillow` — salvar espectrogramas como imagens (entrada do YOLO)
- `ultralytics` — treinar/avaliar os modelos YOLO
- `torch`, `torchvision` — backbone e treinamento do EchoModel
- `requests`, `tqdm` — download dos datasets
- `pyyaml` — gerar o `data.yaml` do YOLO


In [ ]:

# Execute uma vez (remova o # se necessário)
# !pip install --quiet librosa soundfile numpy pandas pillow ultralytics torch torchvision requests tqdm pyyaml

import os
import json
import math
import shutil
import random
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm import tqdm

# ----------------------------------------------------------------------
# Diretórios do projeto
# ----------------------------------------------------------------------
PROJECT_ROOT = Path("./echomodel_project")
RAW_DIR        = PROJECT_ROOT / "01_raw_bbox_datasets"     # datasets de bbox (Cornell)
YOLO_DATA_DIR  = PROJECT_ROOT / "02_yolo_dataset"           # spectrogramas + labels YOLO
YOLO_RUNS_DIR  = PROJECT_ROOT / "03_yolo_runs"              # treinos do YOLO
PSEUDO_DIR     = PROJECT_ROOT / "04_pseudo_labels"          # pseudo-labels nos dados Xeno-Canto
ECHODATA_DIR   = PROJECT_ROOT / "05_echomodel_dataset"      # dataset final do EchoModel
ECHOMODEL_DIR  = PROJECT_ROOT / "06_echomodel_runs"         # treinos do EchoModel

for d in [PROJECT_ROOT, RAW_DIR, YOLO_DATA_DIR, YOLO_RUNS_DIR,
          PSEUDO_DIR, ECHODATA_DIR, ECHOMODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Estrutura de diretórios criada em:", PROJECT_ROOT.resolve())



## 3. Download dos datasets de bounding box (Zenodo)

A função abaixo usa a **API REST do Zenodo** para listar os arquivos de um
registro e baixá-los. Para os datasets desta etapa, o arquivo relevante é
`soundscape_data.zip` (áudio) + `annotations.csv` (+ `species.csv`).

> 💡 Cada `soundscape_data.zip` tem alguns GB. Baixe primeiro **1 ou 2**
> datasets (ex.: `western_us` e `amazon_basin`) para validar o pipeline
> antes de baixar todos.


In [ ]:

ZENODO_API = "https://zenodo.org/api/records/{record_id}"

def download_file(url, dest_path, chunk_size=1 << 20):
    dest_path = Path(dest_path)
    if dest_path.exists():
        print(f"[skip] {dest_path.name} já existe")
        return dest_path
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    total = int(resp.headers.get("content-length", 0))
    with open(dest_path, "wb") as f, tqdm(
        total=total, unit="B", unit_scale=True, desc=dest_path.name
    ) as pbar:
        for chunk in resp.iter_content(chunk_size=chunk_size):
            f.write(chunk)
            pbar.update(len(chunk))
    return dest_path


def download_zenodo_record(record_id, dest_dir, only_files=None, unzip=True):
    \"\"\"
    Baixa arquivos de um registro Zenodo.

    only_files: lista opcional de nomes de arquivo a baixar
                (ex.: ["annotations.csv", "species.csv", "soundscape_data.zip"])
    \"\"\"
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)

    meta = requests.get(ZENODO_API.format(record_id=record_id), timeout=30).json()
    files = meta.get("files", [])

    for f in files:
        fname = f["key"]
        if only_files is not None and fname not in only_files:
            continue
        url = f["links"]["self"]
        out_path = dest_dir / fname
        download_file(url, out_path)

        if unzip and out_path.suffix == ".zip":
            print(f"Extraindo {out_path.name} ...")
            with zipfile.ZipFile(out_path) as zf:
                zf.extractall(dest_dir)

    return dest_dir


# Exemplo: baixar 2 datasets para validar o pipeline ponta a ponta.
DATASETS_TO_DOWNLOAD = ["western_us", "amazon_basin"]

for key in DATASETS_TO_DOWNLOAD:
    info = BBOX_DATASETS[key]
    out_dir = RAW_DIR / key
    print(f"\\n=== {info['name']} (Zenodo {info['zenodo_id']}) ===")
    download_zenodo_record(
        record_id=info["zenodo_id"],
        dest_dir=out_dir,
        only_files=["annotations.csv", "species.csv", "soundscape_data.zip"],
        unzip=True,
    )



## 4. Padronização das anotações em um único DataFrame

Todos os datasets da família "fully-annotated soundscapes" usam o mesmo
esquema de `annotations.csv`:

```
filename, start_time, end_time, low_freq, high_freq, label
```

Vamos carregar todos os `annotations.csv` baixados e juntar em um único
DataFrame com uma coluna extra `dataset` (origem) e o caminho absoluto do
arquivo de áudio correspondente.


In [ ]:

def load_annotations(dataset_key, raw_dir=RAW_DIR):
    ds_dir = Path(raw_dir) / dataset_key
    ann_path = ds_dir / "annotations.csv"
    if not ann_path.exists():
        print(f"[aviso] {ann_path} não encontrado — pulei {dataset_key}")
        return pd.DataFrame()

    df = pd.read_csv(ann_path)

    # Normaliza nomes de colunas (alguns releases variam levemente)
    rename_map = {
        "Begin Time (s)": "start_time",
        "End Time (s)": "end_time",
        "Low Freq (Hz)": "low_freq",
        "High Freq (Hz)": "high_freq",
        "Species eBird Code": "label",
    }
    df = df.rename(columns=rename_map)

    df["dataset"] = dataset_key

    # Caminho do áudio: procura recursivamente pelo nome do arquivo
    audio_index = {}
    for ext in ("*.flac", "*.wav"):
        for p in ds_dir.rglob(ext):
            audio_index[p.name] = p

    df["audio_path"] = df["filename"].map(lambda f: str(audio_index.get(f, "")))
    return df


all_annotations = []
for key in DATASETS_TO_DOWNLOAD:
    df = load_annotations(key)
    print(f"{key}: {len(df)} bounding boxes carregadas")
    all_annotations.append(df)

bbox_df = pd.concat(all_annotations, ignore_index=True)
bbox_df = bbox_df.dropna(subset=["audio_path"])
bbox_df = bbox_df[bbox_df["audio_path"] != ""]

print("\\nTotal de bounding boxes:", len(bbox_df))
print("Espécies únicas:", bbox_df["label"].nunique())
bbox_df.head()



## 5. De áudio + bbox (s, Hz) para espectrograma + label YOLO

### 5.1 Estratégia

1. Cada gravação de 1h é cortada em **janelas (tiles)** de duração fixa
   (`TILE_DURATION`, por padrão 10 s) com sobreposição (`TILE_OVERLAP`).
   Janelas de 10 s dão ao YOLO contexto suficiente para "ver" o formato
   completo das vocalizações, sem gerar imagens gigantes.
2. Para cada tile, geramos um **espectrograma mel em dB** e salvamos como
   imagem PNG (`width x height` fixos).
3. Cada bounding box `(start_time, end_time, low_freq, high_freq)` é
   convertida para coordenadas **normalizadas YOLO**
   `(x_center, y_center, w, h)`, todas em `[0, 1]`, relativas ao tile.
4. Bounding boxes que caem parcialmente fora do tile são **recortadas
   (clipping)**; bboxes com menos de `MIN_BOX_FRACTION` dentro do tile são
   descartadas.
5. Como aqui o objetivo é treinar um **detector tempo-frequência genérico**
   (não um classificador de espécies), usamos **uma única classe** YOLO
   (`bird_call = 0`). Isso simplifica bastante o treino do YOLO e evita o
   desbalanceamento entre espécies nesta etapa.

### 5.2 Mapeamento de eixos

- **Eixo X (tempo)**: `x = tempo_no_tile / TILE_DURATION`
- **Eixo Y (frequência)**: o espectrograma mel é gerado com `np.flipud`,
  de forma que a **linha 0 (topo da imagem) = frequência mais alta** e a
  última linha (base da imagem) = frequência mais baixa — exatamente como
  em uma visualização tradicional de espectrograma. Logo:
  `y = 1 - (freq_hz / FREQ_MAX)`  (e Y cresce "para baixo" na imagem,
  convenção padrão do YOLO).


In [ ]:

import librosa
from PIL import Image

# ----------------------------------------------------------------------
# Parâmetros de geração de espectrograma / tiles
# ----------------------------------------------------------------------
SR            = 32000     # taxa de amostragem alvo
N_FFT         = 1024
HOP_LENGTH    = 320        # ~10 ms por frame em 32kHz
N_MELS        = 256
FREQ_MAX      = SR // 2    # 16000 Hz (Nyquist)

TILE_DURATION = 10.0       # segundos -- tile usado para treinar o YOLO
TILE_OVERLAP  = 2.0        # segundos de sobreposição entre tiles
IMG_WIDTH     = 640         # largura da imagem (px) -- padrão YOLO
IMG_HEIGHT    = 320         # altura da imagem (px)
MIN_BOX_FRACTION = 0.3      # fração mínima do bbox original que precisa
                             # estar dentro do tile para ser mantido


def audio_to_mel_image(y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
                        n_mels=N_MELS, img_w=IMG_WIDTH, img_h=IMG_HEIGHT):
    \"\"\"Converte um trecho de áudio em uma imagem (uint8) de espectrograma mel.\"\"\"
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels,
        fmax=FREQ_MAX,
    )
    S_db = librosa.power_to_db(S, ref=np.max)

    # Normaliza para 0-255
    S_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min() + 1e-8)
    S_norm = (S_norm * 255).astype(np.uint8)

    # linha 0 = frequência mais alta (np.flipud), igual a um espectrograma
    # visual tradicional
    S_img = np.flipud(S_norm)

    img = Image.fromarray(S_img).resize((img_w, img_h), Image.BILINEAR)
    return img


def bbox_to_yolo(start_time, end_time, low_freq, high_freq,
                  tile_start, tile_duration=TILE_DURATION, freq_max=FREQ_MAX):
    \"\"\"
    Converte um bbox em (segundos absolutos, Hz) para coordenadas YOLO
    normalizadas (x_center, y_center, w, h), relativas a um tile que começa
    em `tile_start` e dura `tile_duration` segundos.

    Retorna None se o bbox não estiver (suficientemente) dentro do tile.
    \"\"\"
    tile_end = tile_start + tile_duration

    # --- recorte no tempo ---
    t0 = max(start_time, tile_start)
    t1 = min(end_time, tile_end)
    if t1 <= t0:
        return None  # fora do tile

    original_dur = end_time - start_time
    clipped_dur = t1 - t0
    if original_dur > 0 and (clipped_dur / original_dur) < MIN_BOX_FRACTION:
        return None  # sobrou pouco do bbox original dentro do tile

    # posição relativa ao tile, normalizada [0,1]
    x0 = (t0 - tile_start) / tile_duration
    x1 = (t1 - tile_start) / tile_duration

    # --- recorte na frequência ---
    f0 = max(low_freq, 0.0)
    f1 = min(high_freq, freq_max)
    if f1 <= f0:
        return None

    # eixo Y invertido: 0 Hz -> y=1 (base da imagem), freq_max -> y=0 (topo)
    y0 = 1 - (f1 / freq_max)   # topo do bbox (freq alta)
    y1 = 1 - (f0 / freq_max)   # base do bbox (freq baixa)

    x_center = (x0 + x1) / 2
    y_center = (y0 + y1) / 2
    w = x1 - x0
    h = y1 - y0

    # clipping final de segurança
    x_center = min(max(x_center, 0.0), 1.0)
    y_center = min(max(y_center, 0.0), 1.0)
    w = min(max(w, 1e-3), 1.0)
    h = min(max(h, 1e-3), 1.0)

    return x_center, y_center, w, h


print("Funções de conversão definidas: audio_to_mel_image, bbox_to_yolo")


In [ ]:

# ----------------------------------------------------------------------
# Geração do dataset YOLO (imagens + labels) a partir de bbox_df
# ----------------------------------------------------------------------
YOLO_CLASS_NAMES = ["bird_call"]  # uma única classe nesta etapa


def build_yolo_dataset(bbox_df, out_dir=YOLO_DATA_DIR, tile_duration=TILE_DURATION,
                        tile_overlap=TILE_OVERLAP, max_files_per_dataset=None):
    images_dir = Path(out_dir) / "images"
    labels_dir = Path(out_dir) / "labels"
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    step = tile_duration - tile_overlap
    grouped = bbox_df.groupby(["dataset", "filename", "audio_path"])

    n_done = 0
    files_per_dataset = {}

    for (dataset, filename, audio_path), group in tqdm(grouped, desc="Arquivos"):
        if max_files_per_dataset is not None:
            files_per_dataset.setdefault(dataset, 0)
            if files_per_dataset[dataset] >= max_files_per_dataset:
                continue
            files_per_dataset[dataset] += 1

        y, sr = librosa.load(audio_path, sr=SR, mono=True)
        total_dur = len(y) / sr

        tile_start = 0.0
        tile_idx = 0
        while tile_start < total_dur:
            tile_end = min(tile_start + tile_duration, total_dur)
            if (tile_end - tile_start) < tile_duration * 0.5:
                break  # último pedaço muito curto

            sample_start = int(tile_start * sr)
            sample_end = int(tile_end * sr)
            y_tile = y[sample_start:sample_end]
            if len(y_tile) < int(tile_duration * sr):
                pad = int(tile_duration * sr) - len(y_tile)
                y_tile = np.pad(y_tile, (0, pad))

            # converte bboxes deste arquivo que caem neste tile
            file_boxes = group[
                (group["start_time"] < tile_end) & (group["end_time"] > tile_start)
            ]

            yolo_lines = []
            for _, row in file_boxes.iterrows():
                box = bbox_to_yolo(
                    row["start_time"], row["end_time"],
                    row["low_freq"], row["high_freq"],
                    tile_start=tile_start, tile_duration=tile_duration,
                )
                if box is not None:
                    xc, yc, w, h = box
                    yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

            base_name = f"{dataset}__{Path(filename).stem}__t{int(tile_start)}"
            img = audio_to_mel_image(y_tile)
            img.save(images_dir / f"{base_name}.png")
            with open(labels_dir / f"{base_name}.txt", "w") as f:
                f.write("\\n".join(yolo_lines))

            n_done += 1
            tile_idx += 1
            tile_start += step

    print(f"\\nTiles gerados: {n_done}")
    return images_dir, labels_dir


# ATENÇÃO: para o dataset completo isso pode gerar dezenas de milhares de
# imagens. Use `max_files_per_dataset` para um teste rápido primeiro.
images_dir, labels_dir = build_yolo_dataset(bbox_df, max_files_per_dataset=3)



## 6. Split treino/validação/teste e `data.yaml` do Ultralytics

Dividimos os tiles gerados em treino/val/teste (estratificando por dataset de
origem, para que todas as regiões geográficas apareçam nos três splits) e
geramos o `data.yaml` que o Ultralytics YOLO espera.


In [ ]:

import yaml

def split_yolo_dataset(images_dir=images_dir, labels_dir=labels_dir,
                        out_dir=YOLO_DATA_DIR, val_frac=0.15, test_frac=0.10,
                        seed=42):
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    out_dir = Path(out_dir)

    all_images = sorted(images_dir.glob("*.png"))
    random.Random(seed).shuffle(all_images)

    n = len(all_images)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)
    splits = {
        "val": all_images[:n_val],
        "test": all_images[n_val:n_val + n_test],
        "train": all_images[n_val + n_test:],
    }

    for split, files in splits.items():
        img_out = out_dir / split / "images"
        lbl_out = out_dir / split / "labels"
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path in files:
            lbl_path = labels_dir / (img_path.stem + ".txt")
            shutil.copy(img_path, img_out / img_path.name)
            if lbl_path.exists():
                shutil.copy(lbl_path, lbl_out / lbl_path.name)
        print(f"{split}: {len(files)} imagens")

    data_yaml = {
        "path": str(out_dir.resolve()),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": {i: name for i, name in enumerate(YOLO_CLASS_NAMES)},
    }
    yaml_path = out_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)

    print("\\ndata.yaml escrito em:", yaml_path)
    return yaml_path


data_yaml_path = split_yolo_dataset()



## 7. Treinamento e comparação de vários modelos YOLO

Vamos treinar **várias arquiteturas/tamanhos de YOLO** com a biblioteca
[Ultralytics](https://docs.ultralytics.com/) no mesmo `data.yaml`, e comparar:

- `mAP50`
- `mAP50-95`
- `precision`
- `recall`

para escolher o **melhor detector tempo-frequência**.

Modelos sugeridos (do mais leve ao mais pesado): `yolov8n`, `yolov8s`,
`yolov8m`, `yolo11n`, `yolo11s`. Ajuste a lista conforme a GPU disponível.

> ⚠️ Cada treino pode levar de minutos a horas, dependendo do tamanho do
> dataset e da GPU. Comece com `epochs` baixo (ex.: 10-20) para um
> screening rápido, depois treine o melhor candidato por mais épocas.


In [ ]:

from ultralytics import YOLO

YOLO_VARIANTS = ["yolov8n.pt", "yolov8s.pt", "yolo11n.pt", "yolo11s.pt"]

EPOCHS_SCREENING = 20
IMG_SIZE = (IMG_WIDTH, IMG_HEIGHT)  # (640, 320) -- não-quadrado, ok no YOLOv8+

results_summary = []

for variant in YOLO_VARIANTS:
    run_name = variant.replace(".pt", "")
    print(f"\\n=== Treinando {run_name} ===")

    model = YOLO(variant)
    train_results = model.train(
        data=str(data_yaml_path),
        epochs=EPOCHS_SCREENING,
        imgsz=max(IMG_SIZE),   # ultralytics usa imgsz único; ele ajusta o aspect ratio
        rect=True,             # mantém proporção 640x320 (retangular)
        project=str(YOLO_RUNS_DIR),
        name=run_name,
        exist_ok=True,
    )

    metrics = model.val(data=str(data_yaml_path), split="test")
    results_summary.append({
        "model": run_name,
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
    })

results_df = pd.DataFrame(results_summary).sort_values("mAP50-95", ascending=False)
results_df


In [ ]:

# Seleciona o melhor modelo pelo mAP50-95 no conjunto de teste
best_model_name = results_df.iloc[0]["model"]
best_weights = YOLO_RUNS_DIR / best_model_name / "weights" / "best.pt"
print("Melhor modelo:", best_model_name)
print("Pesos:", best_weights)

best_yolo = YOLO(str(best_weights))



## 8. Pseudo-rotulagem tempo-frequência nos dados do Perch v2 / BirdNET v3

### 8.1 Quais dados são esses?

Perch v2 e BirdNET v3 são treinados predominantemente com:

- **Xeno-Canto** (XC) — milhões de gravações focais de aves, com rótulo de
  espécie por gravação (e às vezes "tipo de vocalização": song/call), mas
  **sem** bounding box de tempo/frequência.
- Coleções complementares (Macaulay Library, iNaturalist Sounds, dados de
  monitoramento acústico, etc.), geralmente também **fracamente rotuladas**.

### 8.2 O que faremos

Para cada gravação dessas fontes:

1. Gerar o espectrograma mel (mesmos parâmetros `SR`, `N_FFT`, `HOP_LENGTH`,
   `N_MELS`, `FREQ_MAX` usados no treino do YOLO), cortado em tiles de
   `TILE_DURATION` segundos (mesmo procedimento da Etapa 5).
2. Rodar `best_yolo` em cada tile → caixas `(x_center, y_center, w, h)`
   normalizadas → converter de volta para `(t_min, t_max) em segundos` e
   `(f_min, f_max) em Hz` (coordenadas absolutas na gravação original).
3. Associar a cada caixa detectada o **target = espécie** vindo do metadado
   original do arquivo (1 target por arquivo nesta primeira etapa —
   trabalhamos com **targets únicos**, como pedido).
4. Quando várias caixas forem detectadas no mesmo arquivo, manter todas
   (cada caixa detectada vira um exemplo `(t_min, t_max, f_min, f_max,
   target)`); a escolha de "qual caixa é a vocalização-alvo" por janela é
   resolvida na Etapa 9, durante a montagem do dataset do EchoModel.
5. Salvar uma tabela unificada `pseudo_labels.csv` com colunas:
   `source_dataset, audio_path, t_min, t_max, f_min, f_max, target, yolo_conf`.

### 8.3 Origem dos metadados de espécie (target)

- Para Xeno-Canto: use a [API do Xeno-Canto](https://xeno-canto.org/explore/api)
  para baixar gravações + metadados (`gen`, `sp`, `en`, `q` = qualidade,
  `type` = song/call, etc.). O campo `target` = `gen + " " + sp` (nome
  científico) ou o código de espécie equivalente ao usado pelo BirdNET/Perch.
- Para outras coleções, use o rótulo de espécie já fornecido pelo dataset
  (ex.: código eBird).

> 💡 Defina `XC_METADATA_CSV` apontando para uma tabela já baixada via API do
> Xeno-Canto (`recording_id/file_path, target`). A função abaixo assume esse
> formato mínimo; adapte conforme sua coleta real.


In [ ]:

def inverse_yolo_to_time_freq(xc, yc, w, h, tile_start,
                                tile_duration=TILE_DURATION, freq_max=FREQ_MAX):
    \"\"\"Converte caixa YOLO normalizada -> (t_min, t_max, f_min, f_max) absolutos.\"\"\"
    x0 = xc - w / 2
    x1 = xc + w / 2
    y0 = yc - h / 2  # topo (freq alta)
    y1 = yc + h / 2  # base (freq baixa)

    t_min = tile_start + x0 * tile_duration
    t_max = tile_start + x1 * tile_duration

    f_max = (1 - y0) * freq_max  # topo -> freq alta
    f_min = (1 - y1) * freq_max  # base -> freq baixa

    return t_min, t_max, f_min, f_max


def pseudo_label_file(audio_path, target, yolo_model=None, conf=0.25,
                        tile_duration=TILE_DURATION, tile_overlap=TILE_OVERLAP):
    \"\"\"
    Roda o YOLO em uma gravação inteira (em tiles) e retorna uma lista de
    dicts {t_min, t_max, f_min, f_max, target, yolo_conf}.
    \"\"\"
    yolo_model = yolo_model or best_yolo
    y, sr = librosa.load(audio_path, sr=SR, mono=True)
    total_dur = len(y) / sr
    step = tile_duration - tile_overlap

    detections = []
    tile_start = 0.0
    while tile_start < total_dur:
        tile_end = min(tile_start + tile_duration, total_dur)
        sample_start = int(tile_start * sr)
        sample_end = int(tile_end * sr)
        y_tile = y[sample_start:sample_end]
        if len(y_tile) < int(tile_duration * sr):
            pad = int(tile_duration * sr) - len(y_tile)
            y_tile = np.pad(y_tile, (0, pad))

        img = audio_to_mel_image(y_tile)
        results = yolo_model.predict(np.array(img.convert("RGB")), conf=conf, verbose=False)

        for r in results:
            for box in r.boxes:
                xc, yc, w, h = box.xywhn[0].tolist()
                t_min, t_max, f_min, f_max = inverse_yolo_to_time_freq(
                    xc, yc, w, h, tile_start=tile_start, tile_duration=tile_duration
                )
                # garante que está dentro dos limites da gravação
                t_min = max(0.0, t_min)
                t_max = min(total_dur, t_max)
                detections.append({
                    "t_min": t_min, "t_max": t_max,
                    "f_min": f_min, "f_max": f_max,
                    "target": target,
                    "yolo_conf": float(box.conf[0]),
                })

        tile_start += step

    return detections


def build_pseudo_label_table(file_target_pairs, source_dataset_name,
                               out_csv=PSEUDO_DIR / "pseudo_labels.csv"):
    \"\"\"
    file_target_pairs: lista de tuplas (audio_path, target)
    \"\"\"
    rows = []
    for audio_path, target in tqdm(file_target_pairs, desc=source_dataset_name):
        dets = pseudo_label_file(audio_path, target)
        for d in dets:
            d["source_dataset"] = source_dataset_name
            d["audio_path"] = str(audio_path)
            rows.append(d)

    df = pd.DataFrame(rows)
    out_csv = Path(out_csv)
    if out_csv.exists():
        df_existing = pd.read_csv(out_csv)
        df = pd.concat([df_existing, df], ignore_index=True)
    df.to_csv(out_csv, index=False)
    print(f"Pseudo-labels salvos em {out_csv} ({len(df)} linhas no total)")
    return df


# Exemplo de uso (ajuste para sua coleta real do Xeno-Canto / BirdNET / Perch):
#
# xc_metadata = pd.read_csv("xeno_canto_metadata.csv")  # colunas: audio_path, target
# pairs = list(zip(xc_metadata["audio_path"], xc_metadata["target"]))
# pseudo_df = build_pseudo_label_table(pairs, source_dataset_name="xeno_canto")
print("Funções de pseudo-rotulagem definidas: pseudo_label_file, build_pseudo_label_table")



## 9. Construção do dataset do EchoModel (janelas de 5 s, overlap de 2,5 s)

Agora juntamos:

- as gravações pseudo-rotuladas (`pseudo_labels.csv`, Etapa 8), e
- as gravações com bbox real (`bbox_df`, Etapas 1-6, que também têm
  tempo+frequência+espécie e podem ser reaproveitadas diretamente — sem
  precisar do YOLO, já que já têm bbox "ground truth"),

em um único formato de **registro de exemplo do EchoModel**:

```
{
  "audio_path": ...,
  "window_start": <s>,     # início da janela de 5s na gravação original
  "window_duration": 5.0,
  "t_min_rel": ...,        # bbox temporal relativo à janela, em [0,1]
  "t_max_rel": ...,
  "f_min": ...,            # bbox em frequência, em Hz (ou normalizado)
  "f_max": ...,
  "target": ...,           # espécie (classe)
}
```

### 9.1 Regras de janelamento

- Cada gravação é cortada em janelas de `WIN_DURATION = 5.0 s`, com passo
  `WIN_STEP = WIN_DURATION - WIN_OVERLAP = 2.5 s` (overlap de 2,5 s).
- Para cada janela, olhamos todas as caixas (reais ou pseudo) cuja
  intersecção temporal com a janela seja `>= MIN_BOX_FRACTION_ECHO`.
- **Targets únicos por janela (1ª etapa, conforme solicitado):** se mais de
  uma caixa cair na mesma janela, escolhemos a caixa com **maior interseção
  temporal** com a janela como o "alvo principal" da janela (e descartamos
  as demais nesta primeira versão). Isso simplifica a saída do modelo para
  `(t_min_rel, t_max_rel, f_min, f_max, target)` — um único bbox + uma única
  classe por janela. (Uma extensão natural depois é permitir múltiplos
  alvos por janela, no estilo de detecção multi-objeto.)


In [ ]:

WIN_DURATION = 5.0
WIN_OVERLAP  = 2.5
WIN_STEP     = WIN_DURATION - WIN_OVERLAP
MIN_BOX_FRACTION_ECHO = 0.3


def boxes_to_echomodel_windows(boxes_df, audio_path, total_duration=None,
                                 win_duration=WIN_DURATION, win_step=WIN_STEP,
                                 min_fraction=MIN_BOX_FRACTION_ECHO):
    \"\"\"
    boxes_df: DataFrame com colunas [t_min, t_max, f_min, f_max, target]
              referentes a UMA gravação (audio_path).
    Retorna lista de dicts, um por janela de 5s, com o alvo "principal".
    Janelas sem nenhuma caixa válida recebem target=None (podem ser usadas
    como exemplos de "background"/sem vocalização, se desejado).
    \"\"\"
    if total_duration is None:
        total_duration = librosa.get_duration(path=audio_path)

    windows = []
    win_start = 0.0
    while win_start < total_duration:
        win_end = min(win_start + win_duration, total_duration)
        if (win_end - win_start) < win_duration * 0.5:
            break

        best_box = None
        best_overlap = 0.0
        for _, row in boxes_df.iterrows():
            inter_start = max(row["t_min"], win_start)
            inter_end = min(row["t_max"], win_end)
            inter = max(0.0, inter_end - inter_start)
            box_dur = row["t_max"] - row["t_min"]
            frac = inter / box_dur if box_dur > 0 else 0.0
            if frac >= min_fraction and inter > best_overlap:
                best_overlap = inter
                best_box = row

        record = {
            "audio_path": str(audio_path),
            "window_start": win_start,
            "window_duration": win_duration,
        }
        if best_box is not None:
            t0 = max(best_box["t_min"], win_start)
            t1 = min(best_box["t_max"], win_end)
            record.update({
                "t_min_rel": (t0 - win_start) / win_duration,
                "t_max_rel": (t1 - win_start) / win_duration,
                "f_min": float(best_box["f_min"]),
                "f_max": float(best_box["f_max"]),
                "target": best_box["target"],
            })
        else:
            record.update({
                "t_min_rel": None, "t_max_rel": None,
                "f_min": None, "f_max": None, "target": None,
            })

        windows.append(record)
        win_start += win_step

    return windows


def build_echomodel_index(boxes_df, out_csv=ECHODATA_DIR / "echomodel_index.csv",
                            drop_background=True):
    \"\"\"
    boxes_df: DataFrame com colunas [audio_path, t_min, t_max, f_min, f_max, target]
              (ground-truth da Etapa 1-6 e/ou pseudo-labels da Etapa 8, já
              combinados).
    \"\"\"
    all_windows = []
    for audio_path, group in tqdm(boxes_df.groupby("audio_path"), desc="Gravações"):
        windows = boxes_to_echomodel_windows(group, audio_path)
        all_windows.extend(windows)

    df = pd.DataFrame(all_windows)
    if drop_background:
        df = df.dropna(subset=["target"])

    out_csv = Path(out_csv)
    df.to_csv(out_csv, index=False)
    print(f"Índice do EchoModel salvo em {out_csv} ({len(df)} janelas)")
    return df


# Exemplo combinando bbox "ground truth" (bbox_df) + pseudo-labels:
gt_boxes = bbox_df.rename(columns={
    "start_time": "t_min", "end_time": "t_max",
    "low_freq": "f_min", "high_freq": "f_max", "label": "target",
})[["audio_path", "t_min", "t_max", "f_min", "f_max", "target"]]

# pseudo_boxes = pd.read_csv(PSEUDO_DIR / "pseudo_labels.csv")[
#     ["audio_path", "t_min", "t_max", "f_min", "f_max", "target"]
# ]
# combined_boxes = pd.concat([gt_boxes, pseudo_boxes], ignore_index=True)

combined_boxes = gt_boxes  # ajuste quando pseudo_labels.csv existir

echo_index_df = build_echomodel_index(combined_boxes)
echo_index_df.head()



## 10. `Dataset` PyTorch para o EchoModel

### 10.1 Frontend alinhado ao Perch v2 / BirdNET v3

Tanto o **Perch v2** quanto o **BirdNET v3** convertem o áudio em um
**espectrograma log-mel** antes do backbone convolucional. O Perch v2
documenta exatamente esses parâmetros (frontend): áudio mono a **32 kHz**,
janelas de **5 s (160.000 amostras)**, janela de análise de **20 ms** com
**hop de 10 ms**, gerando **500 frames** com **128 bandas mel** cobrindo
**60 Hz – 16 kHz**. O BirdNET v3 (developer preview) também migrou para
**32 kHz** (antes era 48 kHz).

Para que o EchoModel seja **comparável e compatível** com esses dois modelos
(mesmas dimensões de entrada, mesma resolução tempo-frequência), usamos o
**mesmo frontend**:

- `ECHO_SR = 32000`
- janela de análise = 20 ms → `ECHO_N_FFT = 640`
- hop = 10 ms → `ECHO_HOP_LENGTH = 320`
- `ECHO_N_MELS = 128`, faixa `60 Hz – 16 kHz`
- janela de 5 s → **500 frames** (igual ao Perch v2)

### 10.2 Saídas (multi-task)

- `target_idx` — classe (espécie), inteiro.
- `bbox_t` — `[t_min_rel, t_max_rel]`, em `[0,1]` (posição temporal dentro
  da janela de 5 s).
- `bbox_f` — `[f_min_norm, f_max_norm]`, em `[0,1]`, normalizado pela faixa
  do frontend (`ECHO_FMIN`–`ECHO_FMAX` = 60 Hz–16 kHz), igual à faixa que o
  espectrograma efetivamente representa.


In [ ]:

# ----------------------------------------------------------------------
# Frontend alinhado ao Perch v2 / BirdNET v3 (32 kHz, 5 s, 128 mels,
# 60 Hz-16 kHz, janela 20 ms / hop 10 ms -> 500 frames)
# ----------------------------------------------------------------------
ECHO_SR = 32000
ECHO_WIN_MS = 20
ECHO_HOP_MS = 10
ECHO_N_FFT = int(ECHO_SR * ECHO_WIN_MS / 1000)        # 640 amostras (20 ms)
ECHO_HOP_LENGTH = int(ECHO_SR * ECHO_HOP_MS / 1000)   # 320 amostras (10 ms)
ECHO_N_MELS = 128                                      # igual ao Perch v2
ECHO_FMIN = 60.0                                       # Perch v2: 60 Hz-16 kHz
ECHO_FMAX = 16000.0

# checagem: 5s @ 32kHz = 160000 amostras -> 160000 / 320 = 500 frames
assert int(WIN_DURATION * ECHO_SR) // ECHO_HOP_LENGTH == 500

print(f"Frontend EchoModel: {ECHO_SR} Hz | {ECHO_N_MELS} mels | "
      f"{ECHO_FMIN:.0f}-{ECHO_FMAX:.0f} Hz | "
      f"{int(WIN_DURATION * ECHO_SR) // ECHO_HOP_LENGTH} frames por janela de {WIN_DURATION}s")


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader


class EchoModelDataset(Dataset):
    def __init__(self, index_df, label2idx=None, sr=ECHO_SR, n_mels=ECHO_N_MELS,
                  n_fft=ECHO_N_FFT, hop_length=ECHO_HOP_LENGTH, win_duration=WIN_DURATION,
                  fmin=ECHO_FMIN, fmax=ECHO_FMAX):
        self.df = index_df.reset_index(drop=True)
        self.sr = sr
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_duration = win_duration
        self.fmin = fmin
        self.fmax = fmax

        if label2idx is None:
            labels = sorted(self.df["target"].unique())
            label2idx = {lab: i for i, lab in enumerate(labels)}
        self.label2idx = label2idx

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        offset = row["window_start"]
        y, _ = librosa.load(
            row["audio_path"], sr=self.sr, mono=True,
            offset=offset, duration=self.win_duration,
        )
        target_len = int(self.win_duration * self.sr)
        if len(y) < target_len:
            y = np.pad(y, (0, target_len - len(y)))

        S = librosa.feature.melspectrogram(
            y=y, sr=self.sr, n_fft=self.n_fft, hop_length=self.hop_length,
            n_mels=self.n_mels, fmin=self.fmin, fmax=self.fmax,
        )
        S_db = librosa.power_to_db(S, ref=np.max)
        S_norm = (S_db - S_db.mean()) / (S_db.std() + 1e-8)
        spec = torch.tensor(S_norm, dtype=torch.float32).unsqueeze(0)  # (1, n_mels, T)

        target_idx = self.label2idx[row["target"]]
        bbox_t = torch.tensor(
            [row["t_min_rel"], row["t_max_rel"]], dtype=torch.float32
        )
        # normaliza a frequência pela faixa do frontend (fmin-fmax), com clipping
        f_min_norm = (row["f_min"] - self.fmin) / (self.fmax - self.fmin)
        f_max_norm = (row["f_max"] - self.fmin) / (self.fmax - self.fmin)
        bbox_f = torch.tensor(
            [min(max(f_min_norm, 0.0), 1.0), min(max(f_max_norm, 0.0), 1.0)],
            dtype=torch.float32,
        )

        return {
            "spec": spec,
            "target": torch.tensor(target_idx, dtype=torch.long),
            "bbox_t": bbox_t,
            "bbox_f": bbox_f,
        }


# Split treino/val do índice do EchoModel
echo_train_df = echo_index_df.sample(frac=0.85, random_state=42)
echo_val_df = echo_index_df.drop(echo_train_df.index)

label2idx = {lab: i for i, lab in enumerate(sorted(echo_index_df["target"].unique()))}
NUM_CLASSES = len(label2idx)
print("Número de classes (espécies):", NUM_CLASSES)

train_dataset = EchoModelDataset(echo_train_df, label2idx=label2idx)
val_dataset = EchoModelDataset(echo_val_df, label2idx=label2idx)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)



## 11. Arquitetura do EchoModel — backbone convolucional (Perch v2 / BirdNET v3) + camada Transformer ("EchoFormer")

### 11.1 Como Perch v2 e BirdNET v3 são construídos

| | **Perch v2** | **BirdNET v3 (preview)** |
|---|---|---|
| Entrada | áudio mono 32 kHz, janelas de 5 s | áudio mono **32 kHz** (mudou de 48 kHz), comprimento variável |
| Frontend | log-mel: 20 ms/10 ms, 128 mels, 60 Hz–16 kHz → 500 frames | espectrograma (histórico: escala linear/mel + blocos EfficientNet) |
| Backbone | **EfficientNet-B3** (~12 M parâmetros), blocos *inverted residual* com convoluções *depthwise-separable* + *squeeze-and-excitation* | blocos **EfficientNet/EfficientNetV2** (*inverted residual*, *depthwise-separable*) |
| Saída intermediária | **"spatial embedding"** não-poolado, formato `(5, 3, 1536)` — um *grid* tempo×frequência de embeddings, antes do pooling global | mapa de features convolucional antes do *global pooling* |
| Cabeça final | classificação softmax (~15.000 classes), aplicada após *pooling* do *spatial embedding* | classificação multi-rótulo |

**Ponto-chave para o EchoModel:** os dois modelos já produzem, internamente,
um **mapa de embeddings tempo-frequência** (o "spatial embedding" do Perch
v2) antes de colapsá-lo num vetor único via *pooling* global. Esse mapa
preserva exatamente a informação de "em que região tempo-frequência" cada
embedding "olhou" — só que **nenhum dos dois modelos explora essa estrutura
espacial na saída**: ambos colapsam tudo num vetor antes da classificação.

### 11.2 A ideia do EchoModel: CNN (estilo Perch/BirdNET) + Transformer ("EchoFormer")

O EchoModel reaproveita o **mesmo tipo de backbone convolucional** (blocos
*inverted-residual* / *depthwise-separable* + *squeeze-and-excitation*, ou
seja, blocos **MBConv**, a base do EfficientNet usado pelos dois modelos) —
mas, em vez de aplicar *global average pooling* diretamente no "spatial
embedding", **trata esse grid tempo-frequência como uma sequência de tokens
e aplica uma camada Transformer sobre ela**. Essa é a parte **inédita/única**
da arquitetura:

1. **`EchoConvBackbone`** — pilha de blocos `MBConv` (estilo EfficientNet,
   como em Perch v2/BirdNET) que reduz o espectrograma `(1, 128, 500)` para
   um *grid* de embeddings `(C, F'=4, T'=16)` — análogo ao *spatial
   embedding* `(5,3,1536)` do Perch v2, mas com nossas próprias dimensões.
2. **`EchoFormer`** — os `F' x T' = 64` vetores desse grid são tratados como
   tokens de uma sequência. Cada token recebe um **embedding posicional 2D
   separável** (uma componente para a posição em frequência + uma para a
   posição em tempo), preservando a noção de "onde" cada token está no
   espectrograma. Dois tokens aprendíveis extras são adicionados:
   - **`[CLS]`** — token de classificação (espécie/`target`);
   - **`[LOC]`** — token de localização tempo-frequência.

   Tudo passa por algumas camadas de **self-attention** (Transformer
   encoder). Diferente da CNN pura (que só relaciona regiões próximas, a
   menos de empilhar muitas camadas), a auto-atenção permite que o modelo
   relacione **regiões distantes do espectrograma diretamente** — útil para
   cantos com harmônicos em faixas de frequência distantes, ou sílabas
   repetidas em momentos diferentes da janela de 5 s.
3. **Cabeças de saída** (multi-task, como antes):
   - `[CLS]` → `cls_head` → logits de espécie (`NUM_CLASSES`).
   - `[LOC]` → `time_head` (2 saídas, `sigmoid`) → `t_min_rel, t_max_rel`
     e `freq_head` (2 saídas, `sigmoid`) → `f_min_norm, f_max_norm`.

### 11.3 Por que isso ajuda (e é "publicável")

- O token `[LOC]` é **forçado, via supervisão**, a "apontar" para a região
  tempo-frequência da vocalização — o **mapa de atenção** desse token sobre
  os 64 tokens do espectrograma pode ser visualizado como um *heatmap*
  diretamente sobre o espectrograma, dando **interpretabilidade nativa**
  (algo que Perch v2/BirdNET v3 não expõem).
- Como o `[CLS]` e o `[LOC]` compartilham as mesmas camadas de atenção sobre
  o mesmo *grid* de tokens, a tarefa de localização funciona como uma forma
  de **atenção supervisionada auxiliar**: o gradiente da perda de bbox
  também ajusta as representações dos tokens do espectrograma usados pelo
  `[CLS]`, o que é exatamente a hipótese central do seu projeto ("ensinar o
  modelo a olhar para a faixa de frequência certa ajuda a classificar a
  espécie").

### 11.4 Função de perda (igual à anterior)

```
loss = w_cls * CE(class_logits, target)
     + w_t   * SmoothL1(bbox_t_pred, bbox_t_true)
     + w_f   * SmoothL1(bbox_f_pred, bbox_f_true)
```

Pesos iniciais sugeridos: `w_cls=1.0, w_t=0.5, w_f=0.5`.


In [ ]:

import torch.nn as nn
import torch.nn.functional as F


# ----------------------------------------------------------------------
# 11.1 Backbone convolucional estilo EfficientNet (blocos MBConv),
#      como usado pelo Perch v2 (EfficientNet-B3) e pelo BirdNET v3
#      (blocos EfficientNet/EfficientNetV2) -- versão compacta para o
#      EchoModel.
# ----------------------------------------------------------------------
class MBConvBlock(nn.Module):
    \"\"\"Bloco invertido-residual com convolução depthwise-separable +
    squeeze-and-excitation (o bloco básico do EfficientNet usado por
    Perch v2 e BirdNET).\"\"\"

    def __init__(self, in_ch, out_ch, stride=1, expand_ratio=4, se_ratio=0.25):
        super().__init__()
        mid_ch = in_ch * expand_ratio

        self.expand = (
            nn.Sequential(
                nn.Conv2d(in_ch, mid_ch, kernel_size=1, bias=False),
                nn.BatchNorm2d(mid_ch),
                nn.SiLU(),
            )
            if expand_ratio != 1
            else nn.Identity()
        )
        if expand_ratio == 1:
            mid_ch = in_ch

        self.depthwise = nn.Sequential(
            nn.Conv2d(mid_ch, mid_ch, kernel_size=3, stride=stride, padding=1,
                       groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.SiLU(),
        )

        se_ch = max(1, int(in_ch * se_ratio))
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(mid_ch, se_ch, kernel_size=1),
            nn.SiLU(),
            nn.Conv2d(se_ch, mid_ch, kernel_size=1),
            nn.Sigmoid(),
        )

        self.project = nn.Sequential(
            nn.Conv2d(mid_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

        self.use_residual = (stride == 1 and in_ch == out_ch)

    def forward(self, x):
        h = self.expand(x)
        h = self.depthwise(h)
        h = h * self.se(h)
        h = self.project(h)
        if self.use_residual:
            h = h + x
        return h


class EchoConvBackbone(nn.Module):
    \"\"\"
    Reduz o espectrograma log-mel (1, 128, 500) para um grid de embeddings
    tempo-frequência (embed_dim, 4, 16) -- 5 reduções por fator 2, análogo
    ao "spatial embedding" (5, 3, 1536) do Perch v2, porém com dimensões
    próprias do EchoModel.

    128 / 2^5 = 4   (frequência)
    500 / 2^5 = 16  (tempo, aproximadamente -- ver bloco de testes)
    \"\"\"

    def __init__(self, embed_dim=192):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )
        self.stage1 = MBConvBlock(32, 48, stride=2)
        self.stage2 = MBConvBlock(48, 96, stride=2)
        self.stage3 = MBConvBlock(96, 144, stride=2)
        self.stage4 = MBConvBlock(144, embed_dim, stride=2)

    def forward(self, x):
        x = self.stem(x)      # (B, 32,  64, 250)
        x = self.stage1(x)    # (B, 48,  32, 125)
        x = self.stage2(x)    # (B, 96,  16,  63)
        x = self.stage3(x)    # (B, 144,  8,  32)
        x = self.stage4(x)    # (B, embed_dim, 4, 16)
        return x  # (B, C, F', T')


# ----------------------------------------------------------------------
# 11.2 EchoFormer: camada Transformer sobre o grid tempo-frequência,
#      com tokens [CLS] (classificação) e [LOC] (localização)
# ----------------------------------------------------------------------
class TransformerEncoderLayerWithAttn(nn.Module):
    \"\"\"Camada de encoder Transformer (pre-norm) que também retorna os
    pesos de atenção -- usados depois para visualizar onde o token [LOC]
    está "olhando" no espectrograma.\"\"\"

    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, embed_dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.norm1(x)
        attn_out, attn_weights = self.attn(h, h, h, need_weights=True,
                                             average_attn_weights=True)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x, attn_weights  # attn_weights: (B, seq_len, seq_len)


class EchoFormer(nn.Module):
    \"\"\"
    Recebe o grid (B, C, F', T') do EchoConvBackbone, achata em uma
    sequência de F'*T' tokens, soma um embedding posicional 2D separável
    (frequência + tempo), prepende os tokens [CLS] e [LOC], e aplica
    `num_layers` camadas de self-attention.
    \"\"\"

    def __init__(self, embed_dim, freq_tokens, time_tokens,
                  num_heads=4, num_layers=3, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.freq_tokens = freq_tokens
        self.time_tokens = time_tokens

        # embeddings posicionais 2D separáveis (frequência + tempo)
        self.freq_pos = nn.Parameter(torch.zeros(freq_tokens, embed_dim))
        self.time_pos = nn.Parameter(torch.zeros(time_tokens, embed_dim))
        nn.init.trunc_normal_(self.freq_pos, std=0.02)
        nn.init.trunc_normal_(self.time_pos, std=0.02)

        # tokens especiais
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.loc_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.loc_token, std=0.02)

        self.layers = nn.ModuleList([
            TransformerEncoderLayerWithAttn(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, feat_map, return_attention=False):
        B, C, Fp, Tp = feat_map.shape
        # (B, C, F', T') -> (B, F'*T', C)
        tokens = feat_map.flatten(2).transpose(1, 2)

        # posição 2D separável: pos[f, t] = freq_pos[f] + time_pos[t]
        pos = (self.freq_pos[:, None, :] + self.time_pos[None, :, :])  # (F', T', C)
        pos = pos.reshape(Fp * Tp, C).unsqueeze(0)  # (1, F'*T', C)
        tokens = tokens + pos

        cls = self.cls_token.expand(B, -1, -1)
        loc = self.loc_token.expand(B, -1, -1)
        x = torch.cat([cls, loc, tokens], dim=1)  # (B, 2 + F'*T', C)

        last_attn = None
        for layer in self.layers:
            x, last_attn = layer(x)

        x = self.norm(x)
        cls_out, loc_out = x[:, 0], x[:, 1]

        if return_attention:
            # atenção do token [LOC] (índice 1) para os tokens do espectrograma
            loc_attn = last_attn[:, 1, 2:].reshape(B, Fp, Tp)
            return cls_out, loc_out, loc_attn
        return cls_out, loc_out


# ----------------------------------------------------------------------
# 11.3 EchoModel completo
# ----------------------------------------------------------------------
class EchoModel(nn.Module):
    def __init__(self, num_classes, embed_dim=192, num_heads=4, num_layers=3):
        super().__init__()
        self.backbone = EchoConvBackbone(embed_dim=embed_dim)
        # 128 mels / 2^5 = 4 ; 500 frames / 2^5 = 16
        self.transformer = EchoFormer(
            embed_dim=embed_dim, freq_tokens=4, time_tokens=16,
            num_heads=num_heads, num_layers=num_layers,
        )

        self.cls_head = nn.Linear(embed_dim, num_classes)
        self.time_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.SiLU(),
            nn.Linear(embed_dim // 2, 2),
        )
        self.freq_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.SiLU(),
            nn.Linear(embed_dim // 2, 2),
        )

    def forward(self, x, return_attention=False):
        feat_map = self.backbone(x)

        if return_attention:
            cls_out, loc_out, loc_attn = self.transformer(feat_map, return_attention=True)
        else:
            cls_out, loc_out = self.transformer(feat_map)
            loc_attn = None

        class_logits = self.cls_head(cls_out)
        bbox_t = torch.sigmoid(self.time_head(loc_out))
        bbox_f = torch.sigmoid(self.freq_head(loc_out))

        if return_attention:
            return class_logits, bbox_t, bbox_f, loc_attn
        return class_logits, bbox_t, bbox_f


def echomodel_loss(class_logits, bbox_t_pred, bbox_f_pred,
                     target, bbox_t_true, bbox_f_true,
                     w_cls=1.0, w_t=0.5, w_f=0.5):
    loss_cls = F.cross_entropy(class_logits, target)
    loss_t = F.smooth_l1_loss(bbox_t_pred, bbox_t_true)
    loss_f = F.smooth_l1_loss(bbox_f_pred, bbox_f_true)
    total = w_cls * loss_cls + w_t * loss_t + w_f * loss_f
    return total, {"cls": loss_cls.item(), "time": loss_t.item(), "freq": loss_f.item()}


device = "cuda" if torch.cuda.is_available() else "cpu"
echo_model = EchoModel(num_classes=NUM_CLASSES).to(device)

# checagem rápida das dimensões (1 amostra dummy, 128 mels x 500 frames)
with torch.no_grad():
    dummy = torch.randn(2, 1, ECHO_N_MELS, 500).to(device)
    logits, bt, bf = echo_model(dummy)
    print("class_logits:", logits.shape)  # (2, NUM_CLASSES)
    print("bbox_t:", bt.shape)            # (2, 2)
    print("bbox_f:", bf.shape)            # (2, 2)

n_params = sum(p.numel() for p in echo_model.parameters())
print(f"\\nParâmetros do EchoModel: {n_params/1e6:.2f}M")
print(echo_model)



## 11.5 Visualizando onde o token `[LOC]` está "olhando"

Como o `EchoFormer` mantém a posição tempo-frequência de cada token (via
`freq_pos`/`time_pos`), o mapa de atenção do token `[LOC]` (forma `(F'=4,
T'=16)`) pode ser **re-escalado e sobreposto ao espectrograma original**
`(128, ~500)` — funcionando como um *heatmap* de "onde o modelo acha que
está a vocalização". Isso é útil para:

- depurar o EchoModel ainda em treinamento (a atenção deveria convergir para
  a região real da vocalização conforme `bbox_t`/`bbox_f` melhoram);
- comparar qualitativamente com as caixas geradas pelo YOLO (Etapas 5-8) —
  se baterem, é um bom sinal de que a supervisão tempo-frequência está
  funcionando.


In [ ]:

import matplotlib.pyplot as plt

def plot_loc_attention(model, spec, ax=None):
    \"\"\"
    spec: tensor (1, n_mels, T) -- um único espectrograma (já normalizado,
    como retornado por EchoModelDataset).
    \"\"\"
    model.eval()
    with torch.no_grad():
        x = spec.unsqueeze(0).to(device)  # (1, 1, n_mels, T)
        _, _, _, loc_attn = model(x, return_attention=True)

    attn = loc_attn[0].cpu().numpy()  # (F'=4, T'=16)
    # upsample para o tamanho do espectrograma original (n_mels, T)
    attn_img = Image.fromarray((attn / attn.max() * 255).astype(np.uint8))
    attn_resized = attn_img.resize((spec.shape[-1], spec.shape[-2]), Image.BILINEAR)
    attn_resized = np.array(attn_resized) / 255.0

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 4))

    ax.imshow(spec[0].cpu().numpy(), origin="lower", aspect="auto", cmap="magma")
    ax.imshow(attn_resized, origin="lower", aspect="auto", cmap="viridis", alpha=0.4)
    ax.set_title("Espectrograma + atenção do token [LOC]")
    ax.set_xlabel("Frames de tempo")
    ax.set_ylabel("Bandas mel")
    return ax


# Exemplo de uso (após treinar pelo menos algumas épocas):
# sample = val_dataset[0]
# plot_loc_attention(echo_model, sample["spec"])
# plt.show()
print("Função de visualização definida: plot_loc_attention")



## 12. Loop de treinamento do EchoModel


In [ ]:

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, n_batches = 0.0, 0
    comp_losses = {"cls": 0.0, "time": 0.0, "freq": 0.0}

    context = torch.enable_grad() if is_train else torch.no_grad()
    with context:
        for batch in loader:
            spec = batch["spec"].to(device)
            target = batch["target"].to(device)
            bbox_t_true = batch["bbox_t"].to(device)
            bbox_f_true = batch["bbox_f"].to(device)

            class_logits, bbox_t_pred, bbox_f_pred = model(spec)
            loss, comps = echomodel_loss(
                class_logits, bbox_t_pred, bbox_f_pred,
                target, bbox_t_true, bbox_f_true,
            )

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()
            for k in comp_losses:
                comp_losses[k] += comps[k]
            n_batches += 1

    avg_loss = total_loss / max(n_batches, 1)
    avg_comps = {k: v / max(n_batches, 1) for k, v in comp_losses.items()}
    return avg_loss, avg_comps


NUM_EPOCHS = 30
optimizer = torch.optim.AdamW(echo_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history = []
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_comps = run_epoch(echo_model, train_loader, optimizer)
    val_loss, val_comps = run_epoch(echo_model, val_loader)
    scheduler.step()

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                     **{f"train_{k}": v for k, v in train_comps.items()},
                     **{f"val_{k}": v for k, v in val_comps.items()}})

    print(f"[{epoch:02d}/{NUM_EPOCHS}] "
          f"train_loss={train_loss:.4f} (cls={train_comps['cls']:.3f}, "
          f"t={train_comps['time']:.3f}, f={train_comps['freq']:.3f}) | "
          f"val_loss={val_loss:.4f} (cls={val_comps['cls']:.3f}, "
          f"t={val_comps['time']:.3f}, f={val_comps['freq']:.3f})")

torch.save(echo_model.state_dict(), ECHOMODEL_DIR / "echomodel_v0.pt")
history_df = pd.DataFrame(history)
history_df.to_csv(ECHOMODEL_DIR / "training_history.csv", index=False)



## 13. Protocolo de avaliação e comparação com Perch v2 / BirdNET v3

Para que a comparação seja justa, avalie o EchoModel **nos mesmos
soundscapes de teste** usados para avaliar Perch v2 e BirdNET v3 — em geral,
os próprios datasets "fully-annotated soundscapes" listados na Seção 1
(quando não usados no treino) e os conjuntos de teste do BirdCLEF.

### Métricas sugeridas

1. **Classificação (espécie)**:
   - `cmAP` (class-mean Average Precision) — métrica padrão do BirdCLEF,
     usada para comparar com BirdNET/Perch.
   - Top-1 / Top-5 accuracy por janela de 5 s.
2. **Detecção tempo-frequência (diferencial do EchoModel)**:
   - IoU médio entre `(bbox_t_pred, bbox_f_pred)` previsto e o bbox real
     (quando disponível, isto é, nos datasets com ground truth da Seção 1).
   - Erro absoluto médio (MAE) de `t_min/t_max` (em segundos) e `f_min/f_max`
     (em Hz).
3. **Eficiência**: tempo de inferência por janela de 5 s, tamanho do modelo
   (MB) — relevante se o EchoModel também visar dispositivos embarcados,
   como BirdNET-Lite.

A célula abaixo esboça as funções de cálculo de IoU tempo-frequência e de
cmAP, para você adaptar ao seu conjunto de teste.


In [ ]:

def time_freq_iou(box_a, box_b):
    \"\"\"
    box_a, box_b: (t_min, t_max, f_min, f_max), todos normalizados na mesma
    escala (ex.: t em [0,1] dentro da janela, f em [0,1] de FREQ_MAX).
    Calcula a IoU 2D da "caixa" tempo-frequência (analogia ao IoU de bbox em
    visão computacional).
    \"\"\"
    t_min_a, t_max_a, f_min_a, f_max_a = box_a
    t_min_b, t_max_b, f_min_b, f_max_b = box_b

    inter_t = max(0.0, min(t_max_a, t_max_b) - max(t_min_a, t_min_b))
    inter_f = max(0.0, min(f_max_a, f_max_b) - max(f_min_a, f_min_b))
    inter_area = inter_t * inter_f

    area_a = (t_max_a - t_min_a) * (f_max_a - f_min_a)
    area_b = (t_max_b - t_min_b) * (f_max_b - f_min_b)
    union = area_a + area_b - inter_area

    return inter_area / union if union > 0 else 0.0


def class_mean_average_precision(y_true, y_score, num_classes):
    \"\"\"
    cmAP simplificado: AP por classe (one-vs-rest) e média entre classes.
    y_true: array (N,) com índices de classe verdadeiros
    y_score: array (N, num_classes) com probabilidades/logits do modelo
    \"\"\"
    from sklearn.metrics import average_precision_score
    aps = []
    for c in range(num_classes):
        y_true_c = (y_true == c).astype(int)
        if y_true_c.sum() == 0:
            continue
        ap = average_precision_score(y_true_c, y_score[:, c])
        aps.append(ap)
    return float(np.mean(aps)) if aps else float("nan")


print("Funções de avaliação definidas: time_freq_iou, class_mean_average_precision")
print("\\nLembre-se de rodar essas métricas sobre um conjunto de teste comum")
print("(mesmos soundscapes) ao avaliar Perch v2 e BirdNET v3, para comparação justa.")



## 14. Roadmap e próximos passos

1. **Escalar a Etapa 1-7**: baixar todos os 6 datasets "fully-annotated
   soundscapes" (Seção 1) + (opcional) Rainforest Connection, gerar o
   dataset YOLO completo e treinar mais variantes (YOLOv9, YOLOv10, YOLO12,
   tamanhos `m`/`l`) com mais épocas.
2. **Validar o YOLO visualmente**: plotar algumas predições sobre os
   espectrogramas para checar se as caixas tempo-frequência fazem sentido
   biologicamente (ex.: faixas de frequência coerentes com o repertório de
   cada espécie).
3. **Coletar os dados do Perch v2 / BirdNET v3** (Xeno-Canto via API +
   outras fontes citadas pelos respectivos papers/repos) e rodar a Etapa 8
   em escala — isso é o passo que mais consome tempo de processamento
   (milhões de gravações).
4. **Targets únicos → targets múltiplos**: depois de validar o EchoModel
   v0 com 1 alvo por janela, evoluir a Etapa 9-11 para suportar **múltiplos
   alvos por janela de 5 s** (formato mais próximo de detecção multi-objeto,
   tipo YOLO, mas operando diretamente sobre o espectrograma de 5 s).
5. **Comparação final**: rodar a Etapa 13 sobre os soundscapes de teste do
   BirdCLEF (mesmos usados para avaliar BirdNET v3 / Perch v2) e publicar a
   tabela comparativa (cmAP, eficiência, e — diferencial do EchoModel — IoU
   tempo-frequência).
6. **Ablation**: treinar uma versão do EchoModel **sem** as cabeças de
   bbox (`w_t = w_f = 0`) para medir o ganho real que a supervisão
   tempo-frequência traz para a classificação — esse é o "achado
   publicável" mencionado no seu projeto.
7. **Ablation da arquitetura**: comparar o `EchoModel` completo
   (`EchoConvBackbone` + `EchoFormer`) contra uma variante que só faz
   *global average pooling* no `EchoConvBackbone` (sem Transformer,
   mais parecido com Perch v2/BirdNET puros) — isso isola o ganho trazido
   especificamente pela camada de atenção.
8. **Escalar o backbone**: a versão atual (`embed_dim=192`, 3 camadas de
   atenção, ~1,8M parâmetros) é deliberadamente pequena para iteração
   rápida. Para a versão "final", aumente `embed_dim`, `num_layers` e o
   número de canais dos blocos `MBConv` (aproximando-se da escala do
   EfficientNet-B3 do Perch v2, ~12M parâmetros no backbone).
9. **Usar `plot_loc_attention`** (Seção 11.5) durante o treinamento para
   checar qualitativamente se a atenção do token `[LOC]` converge para a
   região tempo-frequência correta — e comparar com as caixas do YOLO
   (Etapa 8) como uma forma de validação cruzada.

---

### Resumo das decisões de design tomadas neste notebook

| Decisão | Valor | Observação |
|---|---|---|
| Taxa de amostragem (YOLO) | 32 kHz | padrão BirdNET/Perch |
| `N_FFT` / `HOP_LENGTH` (YOLO) | 1024 / 320 | ~10 ms/frame, alta resolução p/ bbox |
| `N_MELS` (YOLO) | 256 | resolução em frequência p/ detecção |
| Tile p/ treino do YOLO | 10 s, overlap 2 s | contexto suficiente p/ formas de canto |
| Classes do YOLO (Etapa 5-7) | 1 (`bird_call`) | detector tempo-frequência genérico |
| Janela do EchoModel | 5 s, overlap 2,5 s | conforme especificado |
| Alvo por janela (v0) | único (maior interseção temporal) | extensão futura: múltiplos alvos |
| Frontend do EchoModel | 32 kHz, 128 mels, 60 Hz–16 kHz, 20 ms/10 ms → 500 frames | igual ao frontend do Perch v2 |
| Frequência no EchoModel | normalizada por `[ECHO_FMIN, ECHO_FMAX]` | mesma escala do tempo, em [0,1] |
| Backbone do EchoModel | `EchoConvBackbone` (blocos MBConv: depthwise-separable + SE) | mesmo tipo de bloco do EfficientNet (Perch v2/BirdNET) |
| Camada nova do EchoModel | `EchoFormer` (Transformer + pos. 2D + tokens `[CLS]`/`[LOC]`) | diferencial único do EchoModel |
| Saída do `[CLS]` | classificação de espécie | análogo à cabeça de classificação do Perch v2/BirdNET |
| Saída do `[LOC]` | bbox temporal + bbox de frequência (+ mapa de atenção) | inexistente em Perch v2/BirdNET |
